# Benchmark 1 – Replication of MLP for Antimicrobial Resistance Prediction from MALDI-TOF Spectra

## Objective

In this notebook, we implement the first benchmark model of the project:  
the replication of the **Multi-Layer Perceptron (MLP)** described in:

> Astudillo, C. A., López-Cortés, X. A., Ocque, E., & Manríquez-Troncoso, J. M. (2024).  
> *Multi-label classification to predict antibiotic resistance from raw clinical MALDI-TOF mass spectrometry data*.  
> Scientific Reports, 14, 31283.  
> https://doi.org/10.1038/s41598-024-82697-w


## Background

The referenced study proposes a multi-label classification framework to predict antimicrobial resistance (AMR) from MALDI-TOF mass spectrometry data. The authors benchmarked several machine learning algorithms aming which we find Multi-Layer Perceptron (MLP) achieving competitive and often superior performance in terms of Weighted F1-score (WF1), particularly in multi-label scenarios.


# Imports and Configuration

In [3]:
import sys
import os
import importlib
import pickle
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, make_scorer
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from tqdm.auto import tqdm
from joblib import dump, load
from sklearn.metrics import f1_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
import utils.config
importlib.reload(utils.config)
from utils.config import DATASET_ROOT, PICKLE_OUTPUT_DIR, DRIAMS_A_PICKLE
print("Pickle path:", DRIAMS_A_PICKLE)


Pickle path: /export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/DRIAMS_A_AMR_paper_replication.pkl


/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Loading the DRIAMS-A AMR Dataset

We load the previously generated DRIAMS-A pickle file.

The pickle contains:
- Preprocessed MALDI-TOF spectra (`data`)
- Species labels (`label`)
- Metadata (`meta`)
- Antimicrobial resistance matrix (`amr`)
- Antibiotics list (`antibiotics`)


In [4]:
with open(DRIAMS_A_PICKLE, "rb") as f:
    payload = pickle.load(f)

print("Keys in pickle:", payload.keys())

Keys in pickle: dict_keys(['data', 'label', 'meta', 'amr', 'antibiotics'])


In [5]:
X = payload["data"]
y_species = payload["label"]
amr = payload["amr"]
antibiotics = payload["antibiotics"]

print("Spectral data shape:", X.shape)
print("AMR matrix shape:", amr.shape)
print("Number of antibiotics:", len(antibiotics))
print("Antibiotic names:", antibiotics)
print("Unique species:", np.unique(y_species))


Spectral data shape: (14925, 6000)
AMR matrix shape: (14925, 9)
Number of antibiotics: 9
Antibiotic names: ['Oxacillin', 'Clindamycin', 'Fusidic acid', 'Ciprofloxacin', 'Ceftriaxone', 'Piperacillin-Tazobactam', 'Cefepime', 'Imipenem', 'Meropenem']
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']



# 2. Defining Species-Specific Antibiotic Subsets

According to Table 1 of the paper:

- *Staphylococcus aureus* → Oxacillin, Clindamycin, Fusidic acid
- *Escherichia coli* → Ciprofloxacin, Ceftriaxone, Piperacillin-Tazobactam, Cefepime
- *Klebsiella pneumoniae* → Ciprofloxacin, Ceftriaxone, Imipenem, Meropenem
- *Pseudomonas aeruginosa* → Ciprofloxacin, Imipenem, Meropenem

We now:

1. Subset the dataset by species.
2. Select only the relevant antibiotics.
3. Count the number of complete cases (no missing values across required antibiotics).


In [6]:
# Build AMR DataFrame
amr_df = pd.DataFrame(amr, columns=antibiotics)

# Add species column
amr_df["species"] = y_species

amr_df.head()


,Oxacillin,Clindamycin,Fusidic acid,Ciprofloxacin,Ceftriaxone,Piperacillin-Tazobactam,Cefepime,Imipenem,Meropenem,species
0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,Pseudomonas_Aeruginosa
1,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,Pseudomonas_Aeruginosa
2,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,Pseudomonas_Aeruginosa
3,NaN,NaN,NaN,0.0,NaN,1.0,1.0,NaN,1.0,Pseudomonas_Aeruginosa
4,NaN,NaN,NaN,0.0,NaN,1.0,1.0,NaN,1.0,Pseudomonas_Aeruginosa


In [7]:
species_antibiotics = {
    "Staphylococcus_Aureus": [
        "Oxacillin", "Clindamycin", "Fusidic acid"
    ],
    "Escherichia_Coli": [
        "Ciprofloxacin", "Ceftriaxone",
        "Piperacillin-Tazobactam", "Cefepime"
    ],
    "Klebsiella_Pneumoniae": [
        "Ciprofloxacin", "Ceftriaxone",
        "Imipenem", "Meropenem"
    ],
    "Pseudomonas_Aeruginosa": [
        "Ciprofloxacin", "Imipenem", "Meropenem"
    ]
}

summary = []

for species, ab_list in species_antibiotics.items():
    
    df_species = amr_df[amr_df["species"] == species]
    df_ab = df_species[ab_list]
    
    # Complete cases: all required antibiotics tested
    complete_mask = df_ab.notna().all(axis=1)
    n_complete = complete_mask.sum()
    
    summary.append({
        "Species": species,
        "Total isolates": len(df_species),
        "Complete-case isolates": n_complete
    })

summary_df = pd.DataFrame(summary)
summary_df


,Species,Total isolates,Complete-case isolates
0,Staphylococcus_Aureus,3791,3556
1,Escherichia_Coli,4990,4663
2,Klebsiella_Pneumoniae,2869,2813
3,Pseudomonas_Aeruginosa,3275,2262


# 3. Construction of Resistance Patterns (Label Power Set)

Following the methodology described in the paper, we now:

1. Work separately for each bacterial species.
2. Select only complete-case isolates (no missing AMR values).
3. Encode each resistance combination as a single class (Label Power Set).
4. Remove rare resistance patterns (≤ 10 samples).
5. Count the remaining isolates.



In [8]:
species_datasets = {}
summary_patterns = []

for species, ab_list in species_antibiotics.items():
    
    print(f"\nProcessing {species}")
    
    # Subset species
    df_species = amr_df[amr_df["species"] == species].copy()
    
    # Select antibiotics
    df_ab = df_species[ab_list].copy()
    
    # Keep complete cases only
    complete_mask = df_ab.notna().all(axis=1)
    df_ab = df_ab[complete_mask]
    
    print("Complete-case isolates:", len(df_ab))
    
    # Create resistance pattern string (e.g., 0101)
    df_ab["pattern"] = df_ab.astype(int).astype(str).agg("".join, axis=1)
    
    # Count pattern frequencies
    pattern_counts = df_ab["pattern"].value_counts()
    
    print("Number of unique patterns before filtering:", len(pattern_counts))
    
    # Remove rare patterns (≤ 10 samples)
    valid_patterns = pattern_counts[pattern_counts > 10].index
    df_filtered = df_ab[df_ab["pattern"].isin(valid_patterns)]
    
    print("Remaining isolates after removing rare patterns:", len(df_filtered))
    print("Remaining patterns:", df_filtered["pattern"].nunique())
    
    # Store filtered dataset
    species_datasets[species] = df_filtered.copy()
    
    summary_patterns.append({
        "Species": species,
        "Isolates_after_filtering": len(df_filtered),
        "Number_of_patterns": df_filtered["pattern"].nunique()
    })

summary_patterns_df = pd.DataFrame(summary_patterns)
summary_patterns_df



Processing Staphylococcus_Aureus
Complete-case isolates: 3556
Number of unique patterns before filtering: 8
Remaining isolates after removing rare patterns: 3556
Remaining patterns: 8

Processing Escherichia_Coli
Complete-case isolates: 4663
Number of unique patterns before filtering: 13
Remaining isolates after removing rare patterns: 4649
Remaining patterns: 11

Processing Klebsiella_Pneumoniae
Complete-case isolates: 2813
Number of unique patterns before filtering: 9
Remaining isolates after removing rare patterns: 2795
Remaining patterns: 5

Processing Pseudomonas_Aeruginosa
Complete-case isolates: 2262
Number of unique patterns before filtering: 7
Remaining isolates after removing rare patterns: 2257
Remaining patterns: 6


,Species,Isolates_after_filtering,Number_of_patterns
0,Staphylococcus_Aureus,3556,8
1,Escherichia_Coli,4649,11
2,Klebsiella_Pneumoniae,2795,5
3,Pseudomonas_Aeruginosa,2257,6


# 4. Global Stratified Train-Test Split (Based on Resistance Patterns)

To ensure a fair comparison between:

- Multi-label classification (Label Power Set)
- Single-label binary classifiers
- Multiclass classification

we perform a single global train-test split per species.

The split is stratified according to the full resistance pattern,
ensuring that the distribution of resistance combinations is preserved
between train and test sets.

The same split will later be reused for:
- Multi-label training
- All binary antibiotic-specific classifiers
- The multiclass classifier 


In [9]:
global_splits = {}

for species in species_datasets.keys():
    
    print(f"\nSplitting {species}")
    
    df_species = species_datasets[species].copy()
    
    # Extract X indices from original dataset
    indices = df_species.index.values
    
    # Pattern for stratification
    y_pattern = df_species["pattern"].values
    
    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.2,
        stratify=y_pattern,
        random_state=42
    )
    
    global_splits[species] = {
        "train_idx": train_idx,
        "test_idx": test_idx
    }
    
    print("Train size:", len(train_idx))
    print("Test size:", len(test_idx))



Splitting Staphylococcus_Aureus
Train size: 2844
Test size: 712

Splitting Escherichia_Coli
Train size: 3719
Test size: 930

Splitting Klebsiella_Pneumoniae
Train size: 2236
Test size: 559

Splitting Pseudomonas_Aeruginosa
Train size: 1805
Test size: 452


In [10]:
balance_summary = []

for species, ab_list in species_antibiotics.items():
    
    df_species = species_datasets[species]
    test_idx = global_splits[species]["test_idx"]
    
    df_test = df_species.loc[test_idx]
    
    for ab in ab_list:
        
        y_test = df_test[ab]
        
        n_total = len(y_test)
        n_resistant = (y_test == 1).sum()
        n_susceptible = (y_test == 0).sum()
        
        balance_summary.append({
            "Species": species,
            "Antibiotic": ab,
            "Test_total": n_total,
            "Resistant": n_resistant,
            "Susceptible": n_susceptible,
            "Resistant_ratio": n_resistant / n_total if n_total > 0 else np.nan
        })

balance_df = pd.DataFrame(balance_summary)
balance_df


,Species,Antibiotic,Test_total,Resistant,Susceptible,Resistant_ratio
0,Staphylococcus_Aureus,Oxacillin,712,143,569,0.200843
1,Staphylococcus_Aureus,Clindamycin,712,102,610,0.143258
2,Staphylococcus_Aureus,Fusidic acid,712,45,667,0.063202
3,Escherichia_Coli,Ciprofloxacin,930,268,662,0.288172
4,Escherichia_Coli,Ceftriaxone,930,190,740,0.204301
5,Escherichia_Coli,Piperacillin-Tazobactam,930,64,866,0.068817
6,Escherichia_Coli,Cefepime,930,156,774,0.167742
7,Klebsiella_Pneumoniae,Ciprofloxacin,559,100,459,0.178891
8,Klebsiella_Pneumoniae,Ceftriaxone,559,81,478,0.144902
9,Klebsiella_Pneumoniae,Imipenem,559,6,553,0.010733


In [11]:
pattern_balance_results = []

for species in species_datasets.keys():
    
    df_species = species_datasets[species]
    
    train_idx = global_splits[species]["train_idx"]
    test_idx = global_splits[species]["test_idx"]
    
    df_train = df_species.loc[train_idx]
    df_test = df_species.loc[test_idx]
    
    train_pattern_dist = df_train["pattern"].value_counts(normalize=True)
    test_pattern_dist = df_test["pattern"].value_counts(normalize=True)
    
    # Align indices
    combined = pd.DataFrame({
        "Train_ratio": train_pattern_dist,
        "Test_ratio": test_pattern_dist
    }).fillna(0)
    
    combined["Abs_difference"] = abs(combined["Train_ratio"] - combined["Test_ratio"])
    
    print(f"\n=== {species} ===")
    display(combined.sort_values("Abs_difference", ascending=False))



=== Staphylococcus_Aureus ===


,Train_ratio,Test_ratio,Abs_difference
pattern,,,
000,0.683193,0.683989,0.000796
011,0.004923,0.004213,0.000709
100,0.118847,0.119382,0.000535
010,0.080520,0.080056,0.000464
001,0.031294,0.030899,0.000395
101,0.022152,0.022472,0.000320
110,0.053446,0.053371,0.000075
111,0.005626,0.005618,0.000008



=== Escherichia_Coli ===


,Train_ratio,Test_ratio,Abs_difference
pattern,,,
1111,0.021780,0.022581,0.000801
1101,0.106749,0.107527,0.000778
1010,0.014520,0.015054,0.000534
0000,0.646679,0.646237,0.000443
0101,0.031460,0.031183,0.000277
1100,0.025007,0.024731,0.000276
0010,0.020704,0.020430,0.000274
0100,0.007798,0.007527,0.000271
0111,0.006722,0.006452,0.000271



=== Klebsiella_Pneumoniae ===


,Train_ratio,Test_ratio,Abs_difference
pattern,,,
1100,0.100626,0.100179,0.000447
0100,0.033542,0.033989,0.000447
1111,0.010286,0.010733,0.000447
0000,0.787567,0.787120,0.000447
1000,0.067979,0.067979,0.000000



=== Pseudomonas_Aeruginosa ===


,Train_ratio,Test_ratio,Abs_difference
pattern,,,
011,0.058172,0.059735,0.001563
010,0.012188,0.011062,0.001126
110,0.007756,0.008850,0.001093
000,0.803878,0.803097,0.000781
100,0.071468,0.070796,0.000672
111,0.046537,0.046460,0.000077


In [12]:
binary_balance_results = []

for species, ab_list in species_antibiotics.items():
    
    df_species = species_datasets[species]
    
    train_idx = global_splits[species]["train_idx"]
    test_idx = global_splits[species]["test_idx"]
    
    df_train = df_species.loc[train_idx]
    df_test = df_species.loc[test_idx]
    
    for ab in ab_list:
        
        train_res_ratio = (df_train[ab] == 1).mean()
        test_res_ratio = (df_test[ab] == 1).mean()
        
        binary_balance_results.append({
            "Species": species,
            "Antibiotic": ab,
            "Train_resistant_ratio": train_res_ratio,
            "Test_resistant_ratio": test_res_ratio,
            "Absolute_difference": abs(train_res_ratio - test_res_ratio)
        })

binary_balance_df = pd.DataFrame(binary_balance_results)
binary_balance_df.sort_values("Absolute_difference", ascending=False)


,Species,Antibiotic,Train_resistant_ratio,Test_resistant_ratio,Absolute_difference
3,Escherichia_Coli,Ciprofloxacin,0.286636,0.288172,0.001536
13,Pseudomonas_Aeruginosa,Meropenem,0.104709,0.106195,0.001486
12,Pseudomonas_Aeruginosa,Imipenem,0.124654,0.126106,0.001452
1,Staphylococcus_Aureus,Clindamycin,0.144515,0.143258,0.001256
6,Escherichia_Coli,Cefepime,0.166711,0.167742,0.001030
2,Staphylococcus_Aureus,Fusidic acid,0.063994,0.063202,0.000792
0,Staphylococcus_Aureus,Oxacillin,0.200070,0.200843,0.000772
5,Escherichia_Coli,Piperacillin-Tazobactam,0.068298,0.068817,0.000519
8,Klebsiella_Pneumoniae,Ceftriaxone,0.144454,0.144902,0.000447
10,Klebsiella_Pneumoniae,Meropenem,0.010286,0.010733,0.000447


# 5. Benchmark 1 — Single-label (Binary) Classification per Antibiotic

In this experiment, we replicate the paper’s *single-label* benchmark:

- A separate binary classifier is trained **for each species and each antibiotic**.
- The target is binary: `0 = Susceptible (S)`, `1 = Resistant (R)`.
- Hyperparameters are optimized via **Bayesian optimization (BayesSearchCV)** using **Weighted F1 (WF1)** as the objective.
- We reuse the same global train/test split defined previously (stratified by resistance pattern), so that comparisons across tasks remain fair.

For each (species, antibiotic), we will:
1. Build `X_train`, `X_test` from MALDI-TOF spectra.
2. Build `y_train`, `y_test` from the antibiotic-specific AMR labels.
3. Run Bayesian optimization for an MLP classifier.
4. Evaluate WF1 on the held-out test set.

## 5.1 MLP Hyperparameter Optimization Setup

We define an MLP wrapper to expose hidden-layer sizes as tunable parameters.
We then create a BayesSearchCV object with the same search space used in the paper:

- `activation ∈ {identity, logistic, tanh, relu}`
- `solver ∈ {sgd, adam}`
- `alpha ∈ [1e-6, 1e-2]` (log-uniform)
- `learning_rate ∈ {constant, invscaling, adaptive}`
- `layer1, layer2, layer3 ∈ [10, 1000]`

Optimization metric: **Weighted F1 (WF1)**.

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class Identity(nn.Module):
    def forward(self, x):
        return x


class MLPBinary(nn.Module):
    def __init__(self, input_dim, layer1, layer2, layer3, activation):
        super().__init__()

        activations = {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "logistic": nn.Sigmoid(),
            "identity": Identity()
        }

        self.net = nn.Sequential(
            nn.Linear(input_dim, layer1),
            activations[activation],
            nn.Linear(layer1, layer2),
            activations[activation],
            nn.Linear(layer2, layer3),
            activations[activation],
            nn.Linear(layer3, 1)
        )

    def forward(self, x):
        return self.net(x)

Using device: cpu


In [19]:
def optimize_mlp_optuna(X_train, y_train, n_trials=200, n_splits=5):

    input_dim = X_train.shape[1]

    def objective(trial):

        # EXACT ranges from paper
        layer1 = trial.suggest_int("layer1", 10, 500)
        layer2 = trial.suggest_int("layer2", 10, 500)
        layer3 = trial.suggest_int("layer3", 10, 500)

        activation = trial.suggest_categorical(
            "activation", ["identity", "logistic", "tanh", "relu"]
        )

        solver = trial.suggest_categorical("solver", ["adam", "sgd"])

        lr = trial.suggest_float("lr", 1e-6, 1e-2, log=True)

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

        fold_scores = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):

            X_tr = torch.tensor(X_train[tr_idx], dtype=torch.float32).to(device)
            y_tr = torch.tensor(y_train[tr_idx], dtype=torch.float32).view(-1,1).to(device)

            X_val = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
            y_val = torch.tensor(y_train[val_idx], dtype=torch.float32).view(-1,1).to(device)

            train_loader = DataLoader(
                TensorDataset(X_tr, y_tr),
                batch_size=128,
                shuffle=True
            )

            model = MLPBinary(
                input_dim,
                layer1,
                layer2,
                layer3,
                activation
            ).to(device)

            if solver == "adam":
                optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            else:
                optimizer = torch.optim.SGD(model.parameters(), lr=lr)

            criterion = nn.BCEWithLogitsLoss()

            best_val_score = -np.inf
            best_state = None
            patience = 20
            patience_counter = 0

            for epoch in range(1200):  # EXACT from paper

                model.train()
                for xb, yb in train_loader:
                    optimizer.zero_grad()
                    logits = model(xb)
                    loss = criterion(logits, yb)
                    loss.backward()
                    optimizer.step()

                model.eval()
                with torch.no_grad():
                    logits_val = model(X_val)
                    preds = (torch.sigmoid(logits_val) > 0.5).cpu().numpy()
                    val_score = f1_score(
                        y_val.cpu().numpy(),
                        preds,
                        average="weighted"
                    )

                if val_score > best_val_score:
                    best_val_score = val_score
                    best_state = model.state_dict()
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    break

            model.load_state_dict(best_state)
            fold_scores.append(best_val_score)

        return np.mean(fold_scores)

    study = optuna.create_study(direction="maximize")

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True
    )

    return study

## 5.2 Running Binary Benchmarks per Species and Antibiotic

We now run the full single-label benchmark.

For each species:
- We use the filtered dataset (`species_datasets[species]`) that already:
  - contains only complete-case isolates,
  - excludes rare resistance patterns (≤10 samples),
  - includes the `pattern` column for stratification.

For each antibiotic in that species:
- We extract `y_train`, `y_test` from the antibiotic column.
- We optimize an MLP using only the training set (CV within train).
- We evaluate WF1 on the held-out test set.

We store:
- test WF1
- train/test sample sizes and class balance
- best hyperparameters

In [ ]:
MODEL_DIR = os.path.join(PROJECT_ROOT, "saved_models", "benchmark1_mlp_optuna_exact")
os.makedirs(MODEL_DIR, exist_ok=True)

results = []

total_models = sum(len(v) for v in species_antibiotics.values())

with tqdm(total=total_models, desc="Training Exact MLP (Paper)") as pbar:

    for species, ab_list in species_antibiotics.items():

        df_sp = species_datasets[species]
        train_idx = global_splits[species]["train_idx"]
        test_idx = global_splits[species]["test_idx"]

        X_train = X[train_idx]
        X_test  = X[test_idx]

        for ab in ab_list:

            print(f"\n[Exact Paper MLP] {species} | {ab}")

            y_train = df_sp.loc[train_idx, ab].astype(int).values
            y_test  = df_sp.loc[test_idx, ab].astype(int).values

            model_path = os.path.join(
                MODEL_DIR,
                f"{species}_{ab}_best_model.pt"
            )

            study = optimize_mlp_optuna(
                X_train,
                y_train,
                n_trials=200
            )

            best_params = study.best_params

            # Final training on FULL training set
            model = MLPBinary(
                X_train.shape[1],
                best_params["layer1"],
                best_params["layer2"],
                best_params["layer3"],
                best_params["activation"]
            ).to(device)

            if best_params["solver"] == "adam":
                optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
            else:
                optimizer = torch.optim.SGD(model.parameters(), lr=best_params["lr"])

            criterion = nn.BCEWithLogitsLoss()

            X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
            y_tr = torch.tensor(y_train, dtype=torch.float32).view(-1,1).to(device)

            best_state = None
            best_score = -np.inf
            patience = 20
            patience_counter = 0

            for epoch in range(1200):

                model.train()
                optimizer.zero_grad()
                logits = model(X_tr)
                loss = criterion(logits, y_tr)
                loss.backward()
                optimizer.step()

                model.eval()
                with torch.no_grad():
                    preds = (torch.sigmoid(model(X_tr)) > 0.5).cpu().numpy()
                    train_score = f1_score(y_train, preds, average="weighted")

                if train_score > best_score:
                    best_score = train_score
                    best_state = model.state_dict()
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    break

            model.load_state_dict(best_state)

            # Save FINAL trained best model
            torch.save({
                "model_state_dict": model.state_dict(),
                "params": best_params,
                "score": best_score
            }, model_path)

            print("  → Best model saved.")

            # Test evaluation
            model.eval()
            with torch.no_grad():
                X_te = torch.tensor(X_test, dtype=torch.float32).to(device)
                preds = (torch.sigmoid(model(X_te)) > 0.5).cpu().numpy()

            wf1 = f1_score(y_test, preds, average="weighted")

            results.append({
                "Species": species,
                "Antibiotic": ab,
                "WF1_test": wf1
            })

            pbar.update(1)

results_df = pd.DataFrame(results).sort_values(
    ["Species", "WF1_test"],
    ascending=[True, False]
)

results_df

Training Exact MLP (Paper):   0%|          | 0/14 [00:00<?, ?it/s][I 2026-02-20 11:10:48,923] A new study created in memory with name: no-name-ad9a9001-38ff-4c2b-8bd2-9e9779e1672f



[Exact Paper MLP] Staphylococcus_Aureus | Oxacillin



Training Exact MLP (Paper):   0%|          | 0/14 [02:33<?, ?it/s]

[I 2026-02-20 11:13:22,551] Trial 0 finished with value: 0.913588406571271 and parameters: {'layer1': 382, 'layer2': 156, 'layer3': 143, 'activation': 'relu', 'solver': 'adam', 'lr': 2.369886935514959e-05}. Best is trial 0 with value: 0.913588406571271.



Training Exact MLP (Paper):   0%|          | 0/14 [03:02<?, ?it/s]                     

...

[I 2026-02-20 11:18:19,678] Trial 8 finished with value: 0.7128884759701852 and parameters: {'layer1': 83, 'layer2': 310, 'layer3': 369, 'activation': 'identity', 'solver': 'sgd', 'lr': 0.00019036257407242799}. Best is trial 0 with value: 0.913588406571271.



Training Exact MLP (Paper):   0%|          | 0/14 [08:25<?, ?it/s]                    

[I 2026-02-20 11:19:14,489] Trial 9 finished with value: 0.9092681422912674 and parameters: {'layer1': 327, 'layer2': 94, 'layer3': 292, 'activation': 'relu', 'solver': 'adam', 'lr': 0.002279309007170378}. Best is trial 0 with value: 0.913588406571271.



Training Exact MLP (Paper):   0%|          | 0/14 [09:18<?, ?it/s]                     

[I 2026-02-20 11:20:06,925] Trial 10 finished with value: 0.7180697215665524 and parameters: {'layer1': 496, 'layer2': 152, 'layer3': 412, 'activation': 'relu', 'solver': 'adam', 'lr': 1.2207389735966022e-06}. Best is trial 0 with value: 0.913588406571271.


## 5.3 Summary: Mean WF1 Across Antibiotics

To match the paper’s reporting style for single-label benchmarks,
we also compute the mean WF1 across antibiotics within each species.

In [ ]:
species_mean_df = (
    results_df
    .groupby("Species", as_index=False)["WF1_test"]
    .mean()
    .rename(columns={"WF1_test": "Mean_WF1_test"})
    .sort_values("Mean_WF1_test", ascending=False)
)

species_mean_df

In [1]:
import torch
torch.cuda.is_available()

False